In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_huggingface import HuggingFaceEmbeddings 
import chromadb
from langchain_chroma import Chroma
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["HUGINGGFACE_API_KEY"]=os.getenv("HUGINGGFACE_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
loader=WebBaseLoader("https://github.com/shivakumarburugupelli/customer-churn-prediction")
docs=loader.load()
text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=30)
splits=text_splitter.split_documents(docs)

In [56]:
from langchain_community.retrievers import BM25Retriever
bm25_retriever=BM25Retriever.from_documents(splits)
bm25_retriever.k=5

In [57]:
embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3066.62it/s]


In [58]:
vectorstore=Chroma.from_documents(documents=splits, embedding=embeddings,persist_directory="./chroma_db",
    collection_name="github_repo")

In [86]:
query="Why did you remove CustomerID and Surname?"
chroma_retriever=vectorstore.as_retriever(search_kwargs={"k": 5})
vector=chroma_retriever.invoke(query)
bm25=bm25_retriever.invoke(query)
documents=vector+bm25
documents


[Document(id='055c66b1-501b-4713-ad8e-d244ce405c58', metadata={'language': 'en', 'source': 'https://github.com/shivakumarburugupelli/customer-churn-prediction', 'description': 'Customer Churn Prediction using Artificial Neural Network (ANN) with TensorFlow/Keras and Streamlit deployment - shivakumarburugupelli/customer-churn-prediction', 'title': 'GitHub - shivakumarburugupelli/customer-churn-prediction: Customer Churn Prediction using Artificial Neural Network (ANN) with TensorFlow/Keras and Streamlit deployment · GitHub'}, page_content='Do not share my personal information\n      \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n    You can’t perform that action at this time.'),
 Document(id='c6551c4f-95bf-4e65-9dc9-070df2b771d8', metadata={'description': 'Customer Churn Prediction using Artificial Neural Network (ANN) with TensorFlow/Keras and Streamlit deployment - shivakumarburugupelli/customer-churn-prediction', 'title': 'GitHub - shivakumarburugupelli/customer-churn-prediction: Customer Churn P

In [87]:
unique_docs = {}
for doc in documents:
    unique_docs[doc.page_content] = doc
retrieved_docs = list(unique_docs.values())
retrieved_docs=retrieved_docs[:5]

In [88]:
context = "\n\n".join(
    doc.page_content for doc in retrieved_docs)


In [89]:
prompt = f"""
Answer the question using the code context below.

Context:
{context}

Question:
{query}

Answer:
"""


In [90]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

In [91]:
response=llm.invoke(prompt)
print(response.content)

**Answer:**

I removed **CustomerID** and **Surname** because they are *identifiers* that do not carry any predictive information about churn. Keeping them would:

1. **Introduce noise** – The model could learn spurious patterns tied to specific IDs or names rather than genuine customer behavior.
2. **Risk over‑fitting** – The model might memorize individual records instead of learning generalizable relationships.
3. **Compromise privacy** – These columns expose personal data that should not be used in a predictive model or shared in a public repository.

Therefore, they were dropped to keep the feature set clean, focused on meaningful variables, and to protect customer privacy.


In [79]:
import sys
print(sys.executable)

c:\Users\burug\anaconda3\envs\venv\python.exe


In [1]:
import sys
!{sys.executable} -m pip show langchain-huggingface

Name: langchain-huggingface
Version: 1.2.2
Summary: An integration package connecting Hugging Face and LangChain.
Home-page: https://docs.langchain.com/oss/python/integrations/providers/huggingface
Author: 
Author-email: 
License: MIT
Location: c:\Users\burug\anaconda3\envs\myenv\Lib\site-packages
Requires: huggingface-hub, langchain-core, tokenizers
Required-by: 


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings